<a href="https://colab.research.google.com/github/edwardoughton/IGARSS26/blob/main/notebook_2_ai_agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛰️ IGARSS 2026 Summer School 🛰️
## Multi-Temporal Satellite Image Analysis: Tracking NDVI Throughout the Year

**Instructor:** Ed Oughton, George Mason University

**Session duration:** ~60 minutes

---

Welcome to Part 2!

In this notebook we will explore how to perform **multi-temporal analysis** of satellite imagery to:

* Understand how land cover and vegetation change over time
* Process a time series of satellite images (e.g., 12 monthly images)
* Clip imagery to a specific Area of Interest (AOI), such as the George Mason University (GMU) campus
* Calculate and track Normalized Difference Vegetation Index (NDVI) to observe seasonal variations

> **Key message:** Multi-temporal analysis allows us to move beyond static snapshots and understand the dynamic processes shaping our environment throughout the year.

## န Learning Objectives န

By the end of this notebook, you should be able to:

* Search and acquire a time series of Landsat imagery over a specific geographic area (GMU campus)
* Preprocess and clip multiple satellite images to a consistent Area of Interest (AOI)
* Calculate spectral indices like NDVI across multiple time steps
* Visualize seasonal vegetation changes using a multi-temporal stack
* Implement a change detection workflow across an annual cycle
* Interpret phenological patterns from a time series of satellite data

---

## The Power of Multi-Temporal Analysis

Many remote sensing analyses start with a single cloud-free image. While useful for static mapping, a single image misses the dynamic nature of the Earth's surface.

By transitioning to **multi-temporal analysis**, we unlock the ability to observe:

- **Phenology:** The seasonal cycle of vegetation green-up and senescence.
- **Gradual Change:** Slow processes like urban expansion, desertification, or forest degradation.
- **Abrupt Change:** Immediate impacts from events like floods, fires, or rapid deforestation.

In this session, our focus will be on:

1. **Time Series Stacking** — Loading and managing multiple images across a year (e.g., one per month)
2. **Spatial Subsetting** — Clipping our data to a targeted local area (GMU campus) to make processing efficient
3. **Temporal Signatures** — Extracting the NDVI history of specific pixels to understand seasonal patterns
4. **Visualization** — Creating time-series plots and change maps to effectively communicate temporal dynamics

---

## 1. Install and import dependencies

In [1]:
!pip -q install openai pystac-client planetary-computer odc-stac rasterio numpy matplotlib scikit-learn scipy geopandas shapely

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.6/159.6 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 9.3 MB/s eta 0:00:00


In [2]:
import os
import json
import warnings
import textwrap
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import rasterio
from rasterio.plot import show

import pystac_client
import planetary_computer
import odc.stac

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from scipy import ndimage

warnings.filterwarnings('ignore')

DATA_DIR = Path('igarss26_data')
DATA_DIR.mkdir(exist_ok=True)

print('All packages imported.')

All packages imported.


---

## Prompt Engineering for Scientific Code

Before building the pipeline, let's examine **what makes a good AI prompt** for scientific coding.

### The CAPE framework for scientific prompts:

| Element | Description | Example |
|---|---|---|
| **C**ontext | Describe your data and scientific domain | *"I have a 2D numpy array of Landsat NDVI values (float32, range -1 to 1, NaN for nodata)"* |
| **A**ction | State precisely what you want the code to do | *"Classify the NDVI array into 5 land cover classes using K-means"* |
| **P**arameters | Specify constraints, preferences, edge cases | *"Use sklearn, standardize features, set random_state=42, handle NaN by masking"* |
| **E**xpected output | Describe the expected result format | *"Return a 2D integer array with class labels 0–4 and a legend dictionary"* |



---

## 5. Multi-Temporal Change Detection with AI-Assisted Code

Change detection is one of the most impactful applications of multi-temporal satellite imagery. Here we will:

1. Load a **second Landsat scene** from a different season (or year)
2. Use an AI agent to generate the change detection pipeline
3. Critically evaluate and clean up the AI-generated code
4. Visualize vegetation change using **NDVI differencing**

### Change Detection Approaches

| Method | Description | When to use |
|---|---|---|
| **Index differencing** | ΔIndex = Index_t2 − Index_t1 | Quick exploratory analysis |
| **Image differencing** | Per-band subtraction | Multi-band change analysis |
| **CVA (Change Vector Analysis)** | Magnitude + direction of spectral change | Full spectral change characterization |
| **Post-classification comparison** | Classify both dates, compare maps | Categorical land cover change |
| **Bi-temporal deep learning** | CNN/Transformer on image pairs | Complex change patterns |

---

## Critically Evaluating AI-Generated Code

The most important skill for an AI Evangelist is **knowing when to trust AI output and when to push back**.

### Common failure modes for AI geospatial code:

| Failure mode | Example | How to catch it |
|---|---|---|
| **Wrong scaling** | Forgetting to divide Landsat values by 10000 | Check value range after loading |
| **CRS mismatch** | Comparing arrays from different projections | Always check `.crs` attributes |
| **NaN propagation** | Not masking clouds before statistics | Check output for unexpected NaN patterns |
| **Wrong band name** | Using 'B4' vs 'red' for the same band | Read STAC item `assets` dictionary |
| **Off-by-one errors** | Image shape vs. coordinate array length | Print shapes at each step |
| **Silent exceptions** | `try/except` that swallows errors | Use specific exception types |

### A simple code review checklist for AI-generated geospatial code:

1. ☑ Are all input arrays the same shape before operations?
2. ☑ Are reflectance values in the expected range [0, 1]?
3. ☑ Is the CRS consistent throughout?
4. ☑ Are nodata/cloud pixels properly masked before statistics?
5. ☑ Does the output make physical sense (e.g., NDVI > 0.6 in forests, < 0.1 in water)?

In [ ]:
def validate_spectral_array(arr, name, expected_range=(-1, 1)):
    """
    Quick sanity check for spectral index arrays.
    Prints warnings if values fall outside expected bounds.
    """
    valid = arr[np.isfinite(arr)]
    if len(valid) == 0:
        print(f'  ⚠️  {name}: ALL values are NaN!')
        return

    lo, hi = valid.min(), valid.max()
    pct_nan = 100 * np.isnan(arr).mean()

    status = '✅' if expected_range[0] <= lo and hi <= expected_range[1] else '⚠️ '
    print(f'  {status} {name}: range=[{lo:.3f}, {hi:.3f}]  '
          f'mean={valid.mean():.3f}  NaN={pct_nan:.1f}%')

    if lo < expected_range[0] - 0.05 or hi > expected_range[1] + 0.05:
        print(f'       → Expected range {expected_range}. Check scaling!')


print('Validation of spectral arrays:')
validate_spectral_array(ndvi, 'NDVI (summer)')
validate_spectral_array(ndwi, 'NDWI (summer)')
validate_spectral_array(ndbi, 'NDBI (summer)')
validate_spectral_array(ndvi_winter, 'NDVI (winter)')
validate_spectral_array(delta_ndvi, 'ΔNDVI', expected_range=(-2, 2))
validate_spectral_array(red, 'Red reflectance', expected_range=(0, 1))
validate_spectral_array(nir, 'NIR reflectance', expected_range=(0, 1))